# Anexo técnico — Modelado predictivo de Saber 11° a partir de simulacros externos
I.E.T.A. Juan Domínguez Romero de Caracolí · MSc. Jesús Molina Cantillo
Este cuaderno reproduce todas las cifras del anexo: separación Train/Test, prevención de fugas,
preprocesamiento dentro de *pipelines*, entrenamiento, umbrales de decisión, matrices de confusión,
comparación contra línea base y regla experta, y diagnóstico de sobreajuste.

In [3]:
# Solo en Google Colab: subir las dos bases MDT si no están en la carpeta de trabajo
import os
if not os.path.exists('MDT_Simulacros_Saber11_2025_anonimizada.xlsx') and not os.path.exists('../datos'):
    try:
        from google.colab import files
        files.upload()   # seleccione MDT_Simulacros_Saber11_2025_anonimizada.xlsx y MDT_Simulacros_Saber11_2026_anonimizada.xlsx
    except ImportError:
        pass

In [2]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, json, os
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import LinearRegression, RidgeCV, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, RepeatedKFold, GridSearchCV, cross_val_predict, StratifiedKFold
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, roc_auc_score, average_precision_score,
                             confusion_matrix, recall_score, precision_score, f1_score, balanced_accuracy_score, accuracy_score, roc_curve)
SEED=2025; np.random.seed(SEED)
OUT='fig'; os.makedirs(OUT,exist_ok=True)
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.spines.top':False,'axes.spines.right':False})
R={}  # resultados para el documento

ModuleNotFoundError: No module named 'sklearn'

## 1. Carga de las bases MDT anonimizadas y definición de variables

In [3]:
RUTA='../datos/' if os.path.exists('../datos') else './'
fr=[]
for y in (2025,2026):
    d=pd.read_excel(f'{RUTA}MDT_Simulacros_Saber11_{y}_anonimizada.xlsx',sheet_name='MDT'); d['COHORTE']=y
    a=pd.read_excel(f'{RUTA}MDT_Simulacros_Saber11_{y}_anonimizada.xlsx',sheet_name='ALERTAS')
    d['IMP_S1']=d.ID_ESTUDIANTE.isin(a[a.Alerta=='Simulacro 1 imputado'].ID_ESTUDIANTE)
    d['IMP_S2']=d.ID_ESTUDIANTE.isin(a[a.Alerta=='Simulacro 2 imputado'].ID_ESTUDIANTE)
    fr.append(d)
D=pd.concat(fr,ignore_index=True)
AREAS=['LECTURA','MATEMATICAS','SOCIALES','NATURALES','INGLES']
X_AREAS=[f'S1_{a}' for a in AREAS]+[f'S2_{a}' for a in AREAS]
X_GLOB=['S1_GLOBAL','S2_GLOBAL']
TARGET='SB11_GLOBAL'
# Variables excluidas por fuga de etiqueta (construidas con el registro del ICFES) o por redundancia
EXCLUIDAS={'SB11_GLOBAL':'objetivo','EMPAREJAMIENTO_SB11':'fuga (registro ICFES)','ALERTA_SB11_PERFIL_IDENTICO':'fuga (registro ICFES)',
           'ALERTA_SB11_GLOBAL_NO_CUADRA':'fuga (registro ICFES)','EN_ANALISIS_SENSIBILIDAD':'fuga (depende de alertas ICFES)',
           'S2_DESEMPEÑO':'recodificación de S2; solo regla experta','CAMBIO_S2_S1':'combinación lineal de S1 y S2','ID_ESTUDIANTE':'identificador'}
C=D[~(D.IMP_S1|D.IMP_S2)].copy()
print('Casos completos:',len(C),'| con imputados:',len(D))

Casos completos: 97 | con imputados: 112


## 2. Separación Train/Test temporal
Train = cohorte 2025; Test = cohorte 2026. El modelo se ajusta con un año y se evalúa con el siguiente,
como se usaría en la institución. El Test no se toca hasta la evaluación final.

In [4]:
TR=C[C.COHORTE==2025].reset_index(drop=True); TE=C[C.COHORTE==2026].reset_index(drop=True)
R['n_train']=len(TR); R['n_test']=len(TE); R['p_train']=len(TR)/len(C)
# Etiqueta de riesgo: estar en el cuartil inferior de SU PROPIA cohorte (decisión relativa: la institución
# puede dar refuerzo intensivo a cerca de una cuarta parte de cada promoción). La etiqueta solo es salida, nunca predictor.
CORTE=float(TR[TARGET].quantile(0.25)); CORTE_TE=float(TE[TARGET].quantile(0.25))
TR['RIESGO']=(TR[TARGET]<=CORTE).astype(int); TE['RIESGO']=(TE[TARGET]<=CORTE_TE).astype(int)
R['corte']=CORTE; R['corte_te']=CORTE_TE; R['prev_train']=TR.RIESGO.mean(); R['prev_test']=TE.RIESGO.mean()
R['n_riesgo_train']=int(TR.RIESGO.sum()); R['n_riesgo_test']=int(TE.RIESGO.sum())
print(f"Train n={len(TR)} ({R['p_train']:.0%}) · Test n={len(TE)} · corte riesgo ≤ {CORTE:.1f} / {CORTE_TE:.1f} · prevalencia train {R['prev_train']:.2f} test {R['prev_test']:.2f}")

Train n=45 (46%) · Test n=52 · corte riesgo ≤ 195.0 / 203.8 · prevalencia train 0.27 test 0.25


## 3. Multicolinealidad (calculada solo en Train)

In [5]:
def vif(X):
    X=np.asarray(X,float); out=[]
    for j in range(X.shape[1]):
        y=X[:,j]; Z=np.column_stack([np.ones(len(X)),np.delete(X,j,1)])
        b=np.linalg.lstsq(Z,y,rcond=None)[0]; r2=1-((y-Z@b)**2).sum()/((y-y.mean())**2).sum(); out.append(1/(1-r2))
    return out
cm=TR[X_AREAS].corr().abs().values.copy(); np.fill_diagonal(cm,0)
R['max_r_areas']=float(cm.max()); R['vif_areas_max']=float(max(vif(TR[X_AREAS]))); R['vif_areas_min']=float(min(vif(TR[X_AREAS])))
R['r_s1s2_train']=float(TR.S1_GLOBAL.corr(TR.S2_GLOBAL)); R['vif_glob']=float(vif(TR[X_GLOB])[0])
# Correlación de un global con sus propias áreas (por qué no se mezclan globales y áreas)
R['r_glob_area_max']=float(max(abs(TR['S2_GLOBAL'].corr(TR['S2_'+a])) for a in AREAS))
print({k:round(v,3) for k,v in R.items() if k.startswith(('max_r','vif','r_'))})

{'max_r_areas': 0.674, 'vif_areas_max': 2.945, 'vif_areas_min': 1.214, 'r_s1s2_train': 0.552, 'vif_glob': 1.437, 'r_glob_area_max': 0.829}


## 4. Modelos de regresión (objetivo continuo SB11_GLOBAL)
Todo el preprocesamiento (imputación y escalado) vive dentro del *pipeline*: se ajusta solo con Train.

In [6]:
promedio=FunctionTransformer(lambda X: np.asarray(X,float).mean(axis=1,keepdims=True))
cv_tr=RepeatedKFold(n_splits=5,n_repeats=3,random_state=SEED)
REG={
 'Línea base (media de Train)':(X_GLOB,DummyRegressor(strategy='mean')),
 'Regla experta (puntaje S2 bruto)':(['S2_GLOBAL'],None),
 'Lineal · promedio S1-S2':(X_GLOB,make_pipeline(SimpleImputer(strategy='median'),promedio,LinearRegression())),
 'Ridge · 10 áreas':(X_AREAS,make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),RidgeCV(alphas=np.logspace(-2,3,30)))),
 'Random Forest · 10 áreas':(X_AREAS,GridSearchCV(make_pipeline(SimpleImputer(strategy='median'),RandomForestRegressor(n_estimators=200,random_state=SEED,n_jobs=-1)),
        {'randomforestregressor__max_depth':[2,3,None],'randomforestregressor__min_samples_leaf':[1,5]},cv=cv_tr,scoring='neg_mean_absolute_error')),
}
rows=[]; FIT_REG={}
for nm,(cols,m) in REG.items():
    if m is None:
        ptr,pte=TR[cols[0]].values,TE[cols[0]].values
    else:
        m.fit(TR[cols],TR[TARGET]); ptr,pte=m.predict(TR[cols]),m.predict(TE[cols]); FIT_REG[nm]=m
    r=dict(Modelo=nm,MAE_train=mean_absolute_error(TR[TARGET],ptr),MAE_test=mean_absolute_error(TE[TARGET],pte),
           RMSE_test=mean_squared_error(TE[TARGET],pte)**0.5,R2_train=r2_score(TR[TARGET],ptr),R2_test=r2_score(TE[TARGET],pte))
    r['Brecha_MAE']=r['MAE_test']-r['MAE_train']; rows.append(r)
TREG=pd.DataFrame(rows).round(2); print(TREG.to_string(index=False))
lin=FIT_REG['Lineal · promedio S1-S2'][-1]; R['b0']=float(lin.intercept_); R['b1']=float(lin.coef_[0])

                          Modelo  MAE_train  MAE_test  RMSE_test  R2_train  R2_test  Brecha_MAE
     Línea base (media de Train)      36.16     27.68      33.60      0.00    -0.00       -8.48
Regla experta (puntaje S2 bruto)      30.62     33.46      43.57      0.27    -0.69        2.84
         Lineal · promedio S1-S2      19.30     18.06      22.42      0.63     0.55       -1.25
                Ridge · 10 áreas      17.59     16.39      21.73      0.69     0.58       -1.20
        Random Forest · 10 áreas      11.06     15.16      19.80      0.88     0.65        4.10


## 5. Modelos de clasificación (riesgo de quedar en el cuartil inferior)
Clase minoritaria ≈ 25 %: no se usa *accuracy* como criterio. Se reportan AUC-ROC, AUC-PR,
exactitud balanceada, sensibilidad, especificidad, precisión y F1.

In [7]:
cv_cl=RepeatedStratifiedKFold(n_splits=5,n_repeats=3,random_state=SEED)
CLF={
 'Línea base (clase mayoritaria)':(X_GLOB,DummyClassifier(strategy='most_frequent')),
 'Logística · promedio S1-S2':(X_GLOB,make_pipeline(SimpleImputer(strategy='median'),promedio,StandardScaler(),LogisticRegression())),
 'Logística L2 · 10 áreas':(X_AREAS,GridSearchCV(make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LogisticRegression(max_iter=2000)),
        {'logisticregression__C':np.logspace(-2,2,9)},cv=cv_cl,scoring='roc_auc')),
 'Random Forest · 10 áreas':(X_AREAS,GridSearchCV(make_pipeline(SimpleImputer(strategy='median'),RandomForestClassifier(n_estimators=200,class_weight='balanced',random_state=SEED,n_jobs=-1)),
        {'randomforestclassifier__max_depth':[2,3,None],'randomforestclassifier__min_samples_leaf':[1,5]},cv=cv_cl,scoring='roc_auc')),
}
FIT_CL={}; PROB={}
for nm,(cols,m) in CLF.items():
    m.fit(TR[cols],TR.RIESGO); FIT_CL[nm]=m
    PROB[nm]=(m.predict_proba(TR[cols])[:,1],m.predict_proba(TE[cols])[:,1])

### 5.1 Selección de umbrales SOLO con Train (probabilidades fuera de pliegue)
Umbral de decisión (refuerzo sí/no): mínimo costo esperado, con un FN (estudiante en riesgo sin refuerzo)
3 veces más costoso que un FP. Umbral de prioridad ALTA: mínimo número total de errores (costos iguales), más exigente.
ALTA: p ≥ umbral Youden · MEDIA: umbral de decisión ≤ p < Youden · BAJA: p < umbral de decisión.

In [8]:
COSTO_FN,COSTO_FP=3,1
def oof(nm):
    cols,m=CLF[nm]; est=m.best_estimator_ if hasattr(m,'best_estimator_') else m
    return cross_val_predict(est,TR[cols],TR.RIESGO,cv=StratifiedKFold(5,shuffle=True,random_state=SEED),method='predict_proba')[:,1]
def umbral_costo(y,p,cfn=None,cfp=None):
    cfn=COSTO_FN if cfn is None else cfn; cfp=COSTO_FP if cfp is None else cfp
    ts=np.unique(np.r_[p,0,1]); c=[cfn*((p<t)&(y==1)).sum()+cfp*((p>=t)&(y==0)).sum() for t in ts]
    return float(ts[int(np.argmin(c))])
UMB={}; OOF={}
for nm in CLF:
    if nm.startswith('Línea'): UMB[nm]=(0.5,0.5); continue
    po=oof(nm); OOF[nm]=po; y=TR.RIESGO.values
    t_dec=umbral_costo(y,po)                      # umbral de decisión (refuerzo sí/no), costo FN=3, FP=1
    t_eq=umbral_costo(y,po,1,1)                    # umbral de prioridad ALTA: costos iguales (mínimo error total)
    UMB[nm]=(t_dec,max(t_eq,t_dec))
print({k:tuple(round(x,3) for x in v) for k,v in UMB.items()})

{'Línea base (clase mayoritaria)': (0.5, 0.5), 'Logística · promedio S1-S2': (0.232, 0.379), 'Logística L2 · 10 áreas': (0.267, 0.32), 'Random Forest · 10 áreas': (0.419, 0.704)}


### 5.2 Métricas en Train y Test (diagnóstico de sobreajuste)

In [9]:
def met(y,p,t):
    yh=(p>=t).astype(int); tn,fp,fn,tp=confusion_matrix(y,yh,labels=[0,1]).ravel()
    auc=roc_auc_score(y,p) if len(np.unique(p))>1 else 0.5
    return dict(AUC=auc,AUC_PR=average_precision_score(y,p),Exact_bal=balanced_accuracy_score(y,yh),Accuracy=accuracy_score(y,yh),
                Sens=tp/(tp+fn),Espec=tn/(tn+fp),Prec=tp/(tp+fp) if tp+fp else 0.0,F1=f1_score(y,yh,zero_division=0),TP=tp,FP=fp,FN=fn,TN=tn)
rows=[]
for nm in CLF:
    ptr,pte=PROB[nm]; t=UMB[nm][0]
    a=met(TR.RIESGO.values,ptr,t); b=met(TE.RIESGO.values,pte,t)
    rows.append(dict(Modelo=nm,Umbral=t,AUC_train=a['AUC'],AUC_test=b['AUC'],AUCPR_test=b['AUC_PR'],ExBal_test=b['Exact_bal'],Acc_test=b['Accuracy'],
                     Sens_test=b['Sens'],Espec_test=b['Espec'],Prec_test=b['Prec'],F1_test=b['F1'],TP=b['TP'],FP=b['FP'],FN=b['FN'],TN=b['TN']))
# Regla experta: nivel I/1 asignado por la empresa en S2 = riesgo
for nm,df in (('tr',TR),('te',TE)): df['EXPERTA']=(df['S2_DESEMPEÑO']=='I/1').astype(int)
a=met(TR.RIESGO.values,TR.EXPERTA.values.astype(float),0.5); b=met(TE.RIESGO.values,TE.EXPERTA.values.astype(float),0.5)
rows.append(dict(Modelo='Regla experta (nivel I/1 de la empresa)',Umbral=np.nan,AUC_train=a['AUC'],AUC_test=b['AUC'],AUCPR_test=b['AUC_PR'],ExBal_test=b['Exact_bal'],Acc_test=b['Accuracy'],
                 Sens_test=b['Sens'],Espec_test=b['Espec'],Prec_test=b['Prec'],F1_test=b['F1'],TP=b['TP'],FP=b['FP'],FN=b['FN'],TN=b['TN']))
TCL=pd.DataFrame(rows); print(TCL.round(3).to_string(index=False))

                                 Modelo  Umbral  AUC_train  AUC_test  AUCPR_test  ExBal_test  Acc_test  Sens_test  Espec_test  Prec_test  F1_test  TP  FP  FN  TN
         Línea base (clase mayoritaria)   0.500      0.500     0.500       0.250       0.500     0.750      0.000       1.000      0.000    0.000   0   0  13  39
             Logística · promedio S1-S2   0.232      0.823     0.870       0.615       0.833     0.750      1.000       0.667      0.500    0.667  13  13   0  26
                Logística L2 · 10 áreas   0.267      0.864     0.870       0.629       0.808     0.750      0.923       0.692      0.500    0.649  12  12   1  27
               Random Forest · 10 áreas   0.419      0.970     0.870       0.732       0.744     0.731      0.769       0.718      0.476    0.588  10  11   3  28
Regla experta (nivel I/1 de la empresa)     NaN      0.712     0.667       0.339       0.667     0.577      0.846       0.487      0.355    0.500  11  20   2  19


## 6. Traducción a decisiones: prioridades alta, media y baja en Test

In [10]:
ELEG='Logística · promedio S1-S2'
t_bajo,t_you=UMB[ELEG]; t_alto=t_bajo; pte=PROB[ELEG][1]
TE['P_RIESGO']=pte; TE['PRIORIDAD']=np.where(pte>=t_you,'ALTA',np.where(pte>=t_bajo,'MEDIA','BAJA'))
mY=met(TE.RIESGO.values,pte,t_you); R['alta_sens']=mY['Sens']; R['alta_espec']=mY['Espec']; R['alta_prec']=mY['Prec']
TPRI=TE.groupby('PRIORIDAD').agg(n=('RIESGO','size'),en_riesgo=('RIESGO','sum'),SB11_media=(TARGET,'mean')).reindex(['ALTA','MEDIA','BAJA'])
TPRI['pct_riesgo']=TPRI.en_riesgo/TPRI.n; print(TPRI.round(2))
# Probabilidad → puntaje de S1-S2 equivalente (para usar la regla sin software)
lg=FIT_CL[ELEG]; sc=lg[2]; lr=lg[3]
def prom_para(p): z=(np.log(p/(1-p))-lr.intercept_[0])/lr.coef_[0,0]; return float(z*sc.scale_[0]+sc.mean_[0])
R['prom_alta']=prom_para(t_you); R['prom_media']=prom_para(t_bajo); R['t_you']=t_you; R['t_dec']=t_bajo
print('Prioridad ALTA si promedio S1-S2 ≤ %.1f; MEDIA si ≤ %.1f; BAJA en otro caso'%(R['prom_alta'],R['prom_media']))

            n  en_riesgo  SB11_media  pct_riesgo
PRIORIDAD                                       
ALTA       17          9      203.12        0.53
MEDIA       9          4      217.00        0.44
BAJA       26          0      261.15        0.00
Prioridad ALTA si promedio S1-S2 ≤ 182.5; MEDIA si ≤ 202.4; BAJA en otro caso


## 7. Sensibilidad: incluyendo los 15 estudiantes con un simulacro faltante
Los valores imputados en la MDT se vuelven a marcar como faltantes y se imputan DENTRO del pipeline
(IterativeImputer ajustado solo con Train), para que la imputación no use información de Test.

In [11]:
D2=D.copy()
for s,flag in (('S1','IMP_S1'),('S2','IMP_S2')):
    cols=[c for c in D2.columns if c.startswith(s+'_') and c!='S2_DESEMPEÑO']
    D2.loc[D2[flag],cols]=np.nan
TR2=D2[D2.COHORTE==2025]; TE2=D2[D2.COHORTE==2026]
y2tr=(TR2[TARGET]<=TR2[TARGET].quantile(.25)).astype(int); y2te=(TE2[TARGET]<=TE2[TARGET].quantile(.25)).astype(int)
sens=make_pipeline(IterativeImputer(random_state=SEED),promedio,StandardScaler(),LogisticRegression())
sens.fit(TR2[X_GLOB],y2tr); p2=sens.predict_proba(TE2[X_GLOB])[:,1]
m2=met(y2te.values,p2,t_bajo); R['sens_auc']=m2['AUC']; R['sens_n_train']=len(TR2); R['sens_n_test']=len(TE2); R['sens_sens']=m2['Sens']; R['sens_espec']=m2['Espec']
reg2=make_pipeline(IterativeImputer(random_state=SEED),promedio,LinearRegression()).fit(TR2[X_GLOB],TR2[TARGET])
R['sens_mae']=mean_absolute_error(TE2[TARGET],reg2.predict(TE2[X_GLOB]))
print({k:round(v,3) for k,v in R.items() if k.startswith('sens')})

{'sens_auc': 0.857, 'sens_n_train': 57, 'sens_n_test': 55, 'sens_sens': np.float64(0.857), 'sens_espec': np.float64(0.756), 'sens_mae': 18.462}


## 8. Robustez: validación en sentido inverso (Train 2026 → Test 2025)

In [12]:
yi_tr=TE.RIESGO; yi_te=TR.RIESGO
mi=make_pipeline(promedio,StandardScaler(),LogisticRegression()).fit(TE[X_GLOB],yi_tr)
R['inv_auc']=roc_auc_score(yi_te,mi.predict_proba(TR[X_GLOB])[:,1])
INV=[]
from sklearn.base import clone
for nm,(cols,m) in REG.items():
    if m is None: INV.append((nm,mean_absolute_error(TR[TARGET],TR[cols[0]]),np.nan)); continue
    mm=clone(m).fit(TE[cols],TE[TARGET]); mae=mean_absolute_error(TR[TARGET],mm.predict(TR[cols]))
    auc=np.nan
    for nc,(cc,mc) in CLF.items():
        if nc.split(' · ')[-1]==nm.split(' · ')[-1] and not nc.startswith('Línea'):
            mcl=clone(mc).fit(TE[cc],TE.RIESGO); auc=roc_auc_score(TR.RIESGO,mcl.predict_proba(TR[cc])[:,1])
    INV.append((nm,mae,auc))
TINV=pd.DataFrame(INV,columns=['Modelo','MAE_inverso','AUC_inverso']); print(TINV.round(3)); TINV.to_csv('tabla_inversa.csv',index=False)

                             Modelo  MAE_inverso  AUC_inverso
0       Línea base (media de Train)       36.145          NaN
1  Regla experta (puntaje S2 bruto)       30.622          NaN
2           Lineal · promedio S1-S2       23.117        0.823
3                  Ridge · 10 áreas       23.719        0.857
4          Random Forest · 10 áreas       25.967        0.857


## 9. Figuras del anexo

In [13]:
# A1: matriz de correlaciones de predictores (Train)
nm_={**{f'S1_{a}':f'S1 {a.title()[:4]}.' for a in AREAS},**{f'S2_{a}':f'S2 {a.title()[:4]}.' for a in AREAS}}
M=TR[X_AREAS].corr().values
fig,ax=plt.subplots(figsize=(6.4,5.4)); im=ax.imshow(M,cmap='RdBu_r',vmin=-1,vmax=1)
lab=[f'S{k} {n}' for k in (1,2) for n in ['Lect.','Mat.','Soc.','Nat.','Ingl.']]
ax.set_xticks(range(10)); ax.set_xticklabels(lab,rotation=60,ha='right'); ax.set_yticks(range(10)); ax.set_yticklabels(lab)
for i in range(10):
    for j in range(10): ax.text(j,i,f'{M[i,j]:.2f}'.replace('.',','),ha='center',va='center',fontsize=7,color='white' if abs(M[i,j])>.6 else 'black')
ax.spines[:].set_visible(False); fig.colorbar(im,ax=ax,shrink=.75,label='r de Pearson'); fig.tight_layout(); fig.savefig(f'{OUT}/A1_correlacion_predictores.png',dpi=220); plt.close()
# A2: Train vs Test (regresión)
t=TREG[~TREG.Modelo.str.startswith('Regla')]
fig,ax=plt.subplots(figsize=(7,3.6)); x=np.arange(len(t)); w=.38
b1=ax.bar(x-w/2,t.MAE_train,w,label='Train (2025)',color='#9db4c0'); b2=ax.bar(x+w/2,t.MAE_test,w,label='Test (2026)',color='#2a6f97')
for bb in (b1,b2):
    for r in bb: ax.text(r.get_x()+r.get_width()/2,r.get_height()+.5,f'{r.get_height():.1f}'.replace('.',','),ha='center',fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([s.replace(' · ','\n') for s in t.Modelo],fontsize=8.5); ax.set_ylabel('MAE (puntos)'); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(f'{OUT}/A2_train_vs_test.png',dpi=220); plt.close()
# A3: ROC en Test + punto de la regla experta
fig,ax=plt.subplots(figsize=(4.8,4.4))
col={'Logística · promedio S1-S2':'#2a6f97','Logística L2 · 10 áreas':'#6a994e','Random Forest · 10 áreas':'#bc6c25'}
for nm,c in col.items():
    f,tp,_=roc_curve(TE.RIESGO,PROB[nm][1]); auc=f"{roc_auc_score(TE.RIESGO,PROB[nm][1]):.2f}".replace('.',','); ax.plot(f,tp,color=c,lw=2,label=f"{nm} (AUC = {auc})")
e=TCL[TCL.Modelo.str.startswith('Regla')].iloc[0]; ax.plot(1-e.Espec_test,e.Sens_test,'ks',ms=8,label='Regla experta (nivel I/1)')
ax.plot([0,1],[0,1],'--',color='grey',lw=1); ax.set_xlabel('1 − especificidad'); ax.set_ylabel('Sensibilidad'); ax.legend(fontsize=7.5,frameon=False,loc='lower right')
fig.tight_layout(); fig.savefig(f'{OUT}/A3_roc_test.png',dpi=220); plt.close()
# A4: matrices de confusión en Test (modelo elegido vs regla experta)
fig,axs=plt.subplots(1,2,figsize=(7.2,3.3))
for ax,(tit,row) in zip(axs,[(f'Modelo elegido (umbral = {t_alto:.2f})'.replace('.',','),TCL[TCL.Modelo==ELEG].iloc[0]),('Regla experta (nivel I/1)',TCL[TCL.Modelo.str.startswith('Regla')].iloc[0])]):
    m=np.array([[row.TN,row.FP],[row.FN,row.TP]],int); ax.imshow(m,cmap='Blues',vmin=0,vmax=m.max()*1.3)
    for i in range(2):
        for j in range(2): ax.text(j,i,m[i,j],ha='center',va='center',fontsize=14,fontweight='bold')
    ax.set_xticks([0,1]); ax.set_xticklabels(['Sin refuerzo','Refuerzo']); ax.set_yticks([0,1]); ax.set_yticklabels(['No en riesgo','En riesgo'])
    ax.set_xlabel('Decisión'); ax.set_title(tit,fontsize=9.5); ax.spines[:].set_visible(False)
axs[0].set_ylabel('Resultado real en Saber 11°'); fig.tight_layout(); fig.savefig(f'{OUT}/A4_confusion_test.png',dpi=220); plt.close()
# A5: costo esperado vs umbral (Train, fuera de pliegue)
po=OOF[ELEG]; y=TR.RIESGO.values; ts=np.linspace(0.02,0.9,89)
cost=[COSTO_FN*((po<t)&(y==1)).sum()+COSTO_FP*((po>=t)&(y==0)).sum() for t in ts]
sen=[((po>=t)&(y==1)).sum()/y.sum() for t in ts]; esp=[((po<t)&(y==0)).sum()/(y==0).sum() for t in ts]
fig,ax=plt.subplots(figsize=(6.6,4.0)); ax2=ax.twinx(); ax2.spines['right'].set_visible(True)
ax.plot(ts,cost,color='#bc4749',lw=2,label='Costo esperado (FN×3 + FP)'); ax2.plot(ts,sen,color='#2a6f97',ls='--',label='Sensibilidad'); ax2.plot(ts,esp,color='#6a994e',ls=':',lw=2,label='Especificidad')
ax.axvline(t_bajo,color='k',lw=1); ax.axvline(t_you,color='grey',lw=1,ls='-.')
ax.text(t_bajo-.01,min(cost)+1,'decisión\n(costo mínimo)',fontsize=8,ha='right'); ax.text(t_you+.01,min(cost)+1,'prioridad alta\n(costos iguales)',fontsize=8,color='dimgrey')
ax.set_xlabel('Umbral de probabilidad'); ax.set_ylabel('Costo'); ax2.set_ylabel('Proporción')
h1,l1=ax.get_legend_handles_labels(); h2,l2=ax2.get_legend_handles_labels(); ax.legend(h1+h2,l1+l2,fontsize=8,frameon=False,loc='upper center',bbox_to_anchor=(0.5,-0.2),ncol=3)
fig.tight_layout(); fig.savefig(f'{OUT}/A5_umbral_costo.png',dpi=220); plt.close()

In [14]:
TREG.to_csv('tabla_regresion.csv',index=False); TCL.to_csv('tabla_clasificacion.csv',index=False); TPRI.to_csv('tabla_prioridades.csv')
json.dump({k:(float(v) if isinstance(v,(np.floating,float,int,np.integer)) else v) for k,v in R.items()},open('resultados.json','w'),indent=1)
print(json.dumps({k:round(float(v),3) for k,v in R.items()},indent=0))

{
"n_train": 45.0,
"n_test": 52.0,
"p_train": 0.464,
"corte": 195.0,
"corte_te": 203.75,
"prev_train": 0.267,
"prev_test": 0.25,
"n_riesgo_train": 12.0,
"n_riesgo_test": 13.0,
"max_r_areas": 0.674,
"vif_areas_max": 2.945,
"vif_areas_min": 1.214,
"r_s1s2_train": 0.552,
"vif_glob": 1.437,
"r_glob_area_max": 0.829,
"b0": 45.669,
"b1": 0.906,
"alta_sens": 0.692,
"alta_espec": 0.795,
"alta_prec": 0.529,
"prom_alta": 182.459,
"prom_media": 202.434,
"t_you": 0.379,
"t_dec": 0.232,
"sens_auc": 0.857,
"sens_n_train": 57.0,
"sens_n_test": 55.0,
"sens_sens": 0.857,
"sens_espec": 0.756,
"sens_mae": 18.462,
"inv_auc": 0.823
}


## 10. Visualización de las figuras generadas

In [15]:
try:
    from IPython.display import Image, display
    for f in sorted(os.listdir(OUT)):
        print(f); display(Image(filename=os.path.join(OUT,f), width=650))
except ImportError:
    print(sorted(os.listdir(OUT)))

['A1_correlacion_predictores.png', 'A2_train_vs_test.png', 'A3_roc_test.png', 'A4_confusion_test.png', 'A5_umbral_costo.png']


In [16]:
# Descargar resultados (Colab)
import shutil
shutil.make_archive('anexo_resultados','zip','.',)
try:
    from google.colab import files; files.download('anexo_resultados.zip')
except ImportError:
    print('Archivo anexo_resultados.zip creado')

Archivo anexo_resultados.zip creado
